# Spike Encoding of Image Data

Explores how images are converted into spike trains and what the resulting temporal representation looks like. Also demonstrates a single LIF neuron's membrane dynamics.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms

from spikenet.image_to_spike_convertor import ImageToSpikeConvertor
from spikenet.network import Network

## Single LIF Neuron

Simulate one `SpikingDenseLayer` with 2 inputs over 25 time steps. The membrane potential trace shows integrate-and-fire dynamics — each `×` marks an input spike that drives the membrane, and a reset occurs whenever the threshold is crossed.

In [ ]:
from spikenet.layers.spiking_dense import SpikingDenseLayer

neuron = SpikingDenseLayer(in_features=2, out_features=1)
neuron.initialize_parameters()

input_spikes = (torch.rand(1, 25, 2) > 0.8).float()
neuron(input_spikes)

fig, ax = plt.subplots(1, 1, figsize=(16, 2))
neuron.plot_mem(ax)
for t in range(25):
    for n in range(2):
        if input_spikes[0, t, n]:
            ax.plot(t, input_spikes[0, t, n] * float(neuron.mem_rec.max()), "x", color=f"C{n}")
plt.legend(["Membrane potential", "Input spikes"])
plt.show()

## Spike-Encoded MNIST

Subclass `ImageToSpikeConvertor` to convert MNIST images (resized to 16×16) into rate-coded spike trains. The time dimension is flattened into the batch, giving shape `(batch, time_steps, 256)`.

In [ ]:
class MyData(ImageToSpikeConvertor):
    RESIZE_SIZE = (16, 16)

    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            train_data=torchvision.datasets.MNIST(
                root="./~data/mnist",
                train=True,
                transform=torchvision.transforms.Compose(
                    [
                        torchvision.transforms.Resize(self.RESIZE_SIZE),
                        transforms.ToTensor(),
                    ]
                ),
                download=True,
            ),
            test_data=torchvision.datasets.MNIST(
                root="./~data/mnist",
                train=False,
                transform=torchvision.transforms.Compose(
                    [
                        torchvision.transforms.Resize(self.RESIZE_SIZE),
                        transforms.ToTensor(),
                    ]
                ),
            ),
            **kwargs,
        )

    def x_transform(self, x):
        x = super().x_transform(x)
        return x.reshape(-1, self.time_scale, self.RESIZE_SIZE[0] * self.RESIZE_SIZE[1])


data = MyData()
data.describe()

Accumulated spike count per pixel across all time steps — a spatial view of total activity. Bright pixels fired frequently; dark pixels rarely or never.

In [ ]:
test_data = data.sample()
data_x, data_y = test_data

img = data_x.reshape([data.time_scale, *data.RESIZE_SIZE])
img = img.sum(0)
plt.imshow(img)
plt.title(f"Input image - y={data_y}")
plt.show()

Spike activity in successive time windows (each frame sums 2 steps). Shows how pixel activations evolve across the temporal dimension — early windows capture initial firing, later windows show decay.

In [ ]:
fig, axs = plt.subplots(1, 8, figsize=(16, 2))
imgs = data_x.reshape([data.time_scale, *data.RESIZE_SIZE])

T = 2
for t in range(0, data.time_scale, T):
    img = imgs[t : t + T].sum(0)
    ax = axs[t // T]
    ax.imshow(img)
    ax.set_title(f"t={t}")

plt.show()

## Network on Spike-Encoded Data

Build a two-layer SNN to classify the spike-encoded inputs. The network operates directly on the temporal spike representation without any rate-based preprocessing.

In [ ]:
input_size = data.RESIZE_SIZE[0] * data.RESIZE_SIZE[1]
net = (Network().add_layer(SpikingDenseLayer, 50, in_features=input_size).add_layer(SpikingDenseLayer, 10)).build()
net.summary()

In [ ]:
net.forward(data_x.reshape(1, -1, input_size))
net.plot_activity()